# Deploy Classifier Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **5-agent e-commerce assistant** using **Strands Agents GraphBuilder** for deterministic routing across Amazon Bedrock AgentCore runtimes. In this notebook, you deploy the Classifier Agent -- a pure LLM agent that classifies customer intent into BROWSE, ORDER, or RECOMMEND categories.

**Notebook 1 of 5** -- This agent is the entry point for the graph DAG. Every customer query flows through the Classifier first.

## Architecture Overview

This tutorial deploys 5 agents across 5 Amazon Bedrock AgentCore runtimes. The Classifier Agent (highlighted below) is the entry point for the graph DAG:

| Runtime | Agent | Protocol | Tools |
|---------|-------|----------|-------|
| **1** | **Classifier** | **A2A (port 9000)** | **None (pure LLM)** |
| 2 | Product | A2A (port 9000) | HTTP API tools |
| 3 | Order | A2A (port 9000) | DynamoDB via MCP |
| 4 | Recommendation | A2A (port 9000) | None (LLM synthesis) |
| 5 | Graph Orchestrator | HTTP (port 8080) | GraphBuilder + A2A clients |

The Classifier outputs structured JSON that the Graph Orchestrator uses to determine routing:
- **BROWSE** -> Product Agent only
- **ORDER** -> Order Agent only
- **RECOMMEND** -> Product + Order (parallel) -> Recommendation Agent

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- IAM permissions to:
  - Create IAM roles and policies
  - Create Amazon Bedrock AgentCore runtimes
  - Push images to Amazon ECR
  - Read/write AWS Systems Manager Parameter Store

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

NOTEBOOK_DIR = Path.cwd()

from utils import (
    CLASSIFIER_AGENT_NAME,
    CLASSIFIER_ROLE_NAME,
    SSM_CLASSIFIER_AGENT_URL,
    create_agentcore_role,
    store_agent_url,
)

session = boto3.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Agent Name: {CLASSIFIER_AGENT_NAME}")

---
## Step 1: Create Classifier Agent with A2A Protocol

The Classifier Agent is a **pure LLM agent** with no tools. It analyzes customer queries and outputs structured JSON indicating the intent category. The graph orchestrator parses this JSON to determine which downstream agents to invoke.

**Output format:** `{"intent": "BROWSE|ORDER|RECOMMEND", "reasoning": "brief explanation"}`

### Code Structure

The following cell creates `classifier_agent/a2a_server.py`:

| Section | What It Does |
|---------|--------------|
| System Prompt | Constrains output to strict JSON with intent classification |
| Agent Creation | Pure LLM agent with no tools -- classification via prompt engineering |
| A2A Server | Wraps agent for inter-agent communication on port 9000 |
| Health Check | `/ping` endpoint for AgentCore container monitoring |

In [ ]:
%%writefile classifier_agent/a2a_server.py
"""Classifier Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Routes customer requests to the appropriate specialist agent by classifying intent
into BROWSE, ORDER, or RECOMMEND categories using structured JSON output.

Key features:
- A2A protocol server for graph-based multi-agent orchestration via Agent-to-Agent messaging
- Pure LLM classification with no tools -- relies on structured JSON output
- OpenTelemetry instrumentation for AWS X-Ray distributed tracing
"""

import json
import logging
import os

import boto3
import uvicorn
from fastapi import FastAPI
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

StrandsTelemetry().setup_otlp_exporter()

PORT = 9000
SSM_CLASSIFIER_AGENT_URL = "/ecommerce-graph/classifier-agent-url"

SYSTEM_PROMPT = """You are a Classifier Agent for an e-commerce assistant.

Your ONLY job is to classify the customer's intent into exactly one of three categories.
You must respond with ONLY a JSON object, no other text.

Intent definitions:
- BROWSE: The customer wants to search, browse, or view products from the catalog.
- ORDER: The customer wants to check order status, order history, or shipment tracking.
- RECOMMEND: The customer wants product recommendations based on their purchase history AND the product catalog.

Output format (strict JSON, no markdown, no explanation outside the JSON):
{"intent": "BROWSE|ORDER|RECOMMEND", "reasoning": "brief explanation"}

Examples:
- "Show me laptops under $1000" -> {"intent": "BROWSE", "reasoning": "Customer wants to search the product catalog for laptops within a price range"}
- "Where is my order?" -> {"intent": "ORDER", "reasoning": "Customer is asking about order status or tracking"}
- "What should I buy next?" -> {"intent": "RECOMMEND", "reasoning": "Customer wants personalized product recommendations based on purchase history"}
- "I bought a camera last month, what accessories would go with it?" -> {"intent": "RECOMMEND", "reasoning": "Customer wants recommendations based on a previous purchase"}
- "Do you have wireless headphones?" -> {"intent": "BROWSE", "reasoning": "Customer wants to search the catalog for a specific product type"}
- "Show me my recent orders" -> {"intent": "ORDER", "reasoning": "Customer wants to view their order history"}
"""


class ToolLoggingHandler:
    """Log agent lifecycle events to Amazon CloudWatch Logs for debugging.

    Although the classifier agent uses no tools, this handler logs completion events
    to provide consistent observability across all agents in the multi-agent system.
    De-duplicates tool invocations by tracking toolUseId values.
    """

    def __init__(self):
        self.logged_tool_ids = set()
        self.tool_count = 0

    def __call__(self, **kwargs):
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_use.get('name', 'Unknown')} ===")
                            input_str = json.dumps(tool_use.get("input", {}))
                            if len(input_str) > 2000:
                                input_str = input_str[:2000] + "..."
                            logger.info(f"TOOL INPUT: {input_str}")
        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# Get AWS region from boto3 session or environment variable
session = boto3.Session()
region = session.region_name or os.environ.get("AWS_REGION", "us-west-2")

classifier_agent = Agent(
    name="Ecommerce_Graph_Classifier",
    description="Intent classification agent for e-commerce assistant routing",
    system_prompt=SYSTEM_PROMPT,
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[],
    callback_handler=ToolLoggingHandler(),
)

# FastAPI app with health check endpoint for Amazon Bedrock AgentCore container monitoring
app = FastAPI()


@app.get("/ping")
async def health():
    return {"status": "healthy"}


# A2A server wraps the agent for inter-agent communication
a2a_server = A2AServer(agent=classifier_agent, serve_at_root=True)
a2a_server.setup(app)

if __name__ == "__main__":
    logger.info(f"Starting Classifier Agent on port {PORT}")
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile classifier_agent/requirements.txt
strands-agents[a2a,otel]
strands-agents-tools
fastapi
uvicorn
boto3

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline:

1. **Creates IAM role** -- Grants permissions for ECR image pull, Bedrock model invocation, and CloudWatch logging
2. **Configures runtime** -- Packages agent code into a deployable container configuration
3. **Launches runtime** -- Builds Docker image, pushes to ECR, and creates the AgentCore runtime

### Create IAM Role and Configure Runtime

The IAM execution role allows the AgentCore runtime to pull container images, invoke Bedrock models, and write logs.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server |
| `protocol` | `A2A` | Enables Agent-to-Agent communication on port 9000 |
| `agent_name` | `ecommerce_graph_classifier` | Unique identifier for this runtime |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

classifier_role_arn = create_agentcore_role(CLASSIFIER_ROLE_NAME, account_id, region)
print(f"IAM Role ARN: {classifier_role_arn}")

classifier_agent_dir = NOTEBOOK_DIR / "classifier_agent"
os.chdir(classifier_agent_dir)

classifier_runtime = Runtime()
classifier_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=classifier_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=CLASSIFIER_AGENT_NAME,
    protocol="A2A",
)

os.chdir(NOTEBOOK_DIR)
print(f"Runtime configured: {CLASSIFIER_AGENT_NAME}")

### Fix Dockerfile Permissions

AgentCore containers run as the `bedrock_agentcore` user, not root. The auto-generated Dockerfile uses `COPY . .` which preserves host file ownership, causing permission errors. This cell updates it to `COPY --chown=bedrock_agentcore:bedrock_agentcore . .`.

In [ ]:
# # Fix Dockerfile COPY ownership
# dockerfile_path = "Dockerfile"
# with open(dockerfile_path, 'r') as f:
#     content = f.read()
# if 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .' not in content:
#     content = content.replace('COPY . .', 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .')
#     with open(dockerfile_path, 'w') as f:
#         f.write(content)
#     print("Dockerfile updated with correct ownership")
# else:
#     print("Dockerfile already has correct ownership")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime. The `auto_update_on_conflict=True` flag updates an existing runtime if one already exists with the same name.

**Note:** First deployment takes 5-10 minutes (subsequent updates are faster).

In [ ]:
print("Launching Classifier Agent (this may take several minutes)...")
os.chdir(classifier_agent_dir)
classifier_launch = classifier_runtime.launch(auto_update_on_conflict=True)
print(f"Classifier Agent ARN: {classifier_launch.agent_arn}")
CLASSIFIER_AGENT_ARN = classifier_launch.agent_arn
os.chdir(NOTEBOOK_DIR)

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN. This URL is the HTTPS endpoint that the Graph Orchestrator uses to send A2A messages to this agent.

In [ ]:
os.chdir(classifier_agent_dir)
status_response = classifier_runtime.status()
status = status_response.endpoint.get("status", "")
classifier_agent_url = None

print(f"Classifier Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    classifier_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Runtime URL: {classifier_agent_url}")
else:
    print(f"Agent not ready. Current status: {status}")

os.chdir(NOTEBOOK_DIR)

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Graph Orchestrator** (Notebook 5) retrieves it to configure `A2AClientToolProvider` for the classifier node

In [ ]:
if status.upper() in ["ACTIVE", "READY"]:
    result = store_agent_url(
        param_name=SSM_CLASSIFIER_AGENT_URL,
        url=classifier_agent_url,
        region=region,
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

In [ ]:
print("=" * 60)
print("Classifier Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {CLASSIFIER_AGENT_NAME}")
print(f"Agent ARN: {CLASSIFIER_AGENT_ARN}")
print(f"IAM Role: {classifier_role_arn}")
print(f"Runtime URL: {classifier_agent_url or 'Not available - agent not ready'}")
print(f"SSM Parameter: {SSM_CLASSIFIER_AGENT_URL}")
print("=" * 60)

---
## Step 3: Test Classifier Agent

Verify the agent works standalone before integrating with the Graph Orchestrator. The `Runtime.invoke()` method calls the AgentCore runtime directly.

**Example queries to try:**
- `"Show me laptops under $1000"` -- should classify as BROWSE
- `"Where is my order?"` -- should classify as ORDER
- `"What should I buy based on my history?"` -- should classify as RECOMMEND

In [ ]:
test_message = "Show me laptops under $1000"

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}],
        }
    },
}

os.chdir(classifier_agent_dir)
print(f"Testing: '{test_message}'")
print("-" * 40)
response = classifier_runtime.invoke(payload, session_id=str(uuid4()))
print(f"\nResponse:\n{response}")
os.chdir(NOTEBOOK_DIR)

### Verify Agent Card (A2A Discovery)

The `A2AServer` wrapper automatically generates an A2A-compliant agent card from your Strands agent. This card is served at `/.well-known/agent-card.json` and enables other agents to discover this agent's capabilities at runtime.

In [ ]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests
import json

agent_card_url = f"{classifier_agent_url}/.well-known/agent-card.json"
credentials = boto3.Session().get_credentials()
request = AWSRequest(method="GET", url=agent_card_url)
SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(request)

response = requests.get(
    agent_card_url,
    headers=dict(request.headers),
)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    card = response.json()
    print(json.dumps(card, indent=2))

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **2. Deploy Product Agent** | Product catalog search with custom HTTP API tools |
| **3. Deploy Order Agent** | DynamoDB integration with MCP tools and async lifespan pattern |
| **4. Deploy Recommendation Agent** | LLM synthesis agent that generates personalized recommendations |
| **5. Deploy Graph Orchestrator** | GraphBuilder DAG that routes through all agents with conditional edges |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
print("Destroying Classifier Agent...")
os.chdir(classifier_agent_dir)
try:
    classifier_runtime.destroy(delete_ecr_repo=True)
    print("Classifier Agent destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)

In [ ]:
ssm = boto3.client("ssm", region_name=region)
try:
    ssm.delete_parameter(Name=SSM_CLASSIFIER_AGENT_URL)
    print(f"Deleted SSM parameter: {SSM_CLASSIFIER_AGENT_URL}")
except Exception as e:
    print(f"Error deleting SSM parameter: {e}")

# Delete auto-generated toolkit files
print("Cleaning up auto-generated files...")
for cleanup_file in ["Dockerfile", ".dockerignore"]:
    cleanup_path = classifier_agent_dir / cleanup_file
    if cleanup_path.exists():
        cleanup_path.unlink()
        print(f"  Deleted: {cleanup_file}")